In [1]:
import torch
import torch.nn.functional as F

In [2]:
words = open("names.txt", "r").read().splitlines()

chars = sorted(list(set("".join(words))))

stoi = {ch: i + 1 for i, ch in enumerate(chars)}
stoi["."] = 0

itos = {i: ch for ch, i in stoi.items()}

VOCAB_SIZE = len(stoi)



xs = []
ys = []

for word in words:
    
    chs = ["."] + list(word) + ["."]
    
    for ch1, ch2 in zip(chs, chs[1:]):
        xs.append(stoi[ch1])
        ys.append(stoi[ch2])

xs = torch.tensor(xs)
ys = torch.tensor(ys)

num = xs.nelement()

print("training pairs:", num)

training pairs: 228146


In [3]:
g = torch.Generator().manual_seed(2147483647)

W = torch.randn((VOCAB_SIZE, VOCAB_SIZE), generator=g, requires_grad=True)

In [4]:
for step in range(100):

    logits = W[xs]                 # row lookup

    loss = F.cross_entropy(logits, ys)

    # L2 regularization
    loss = loss + 0.01 * (W ** 2).mean()

    W.grad = None
    loss.backward()

    W.data += -50 * W.grad

    if step % 10 == 0:
        print(f"step {step:3d} | loss {loss.item():.4f}")


step   0 | loss 3.7686
step  10 | loss 2.6965
step  20 | loss 2.5823
step  30 | loss 2.5414
step  40 | loss 2.5213
step  50 | loss 2.5099
step  60 | loss 2.5027
step  70 | loss 2.4979
step  80 | loss 2.4944
step  90 | loss 2.4919


In [5]:
def sample_names(temp, total=5):
    
    print(f"\nTemperature = {temp}")
    print("-" * 25)

    for _ in range(total):

        out = []
        ix = 0

        while True:

            logits = W[ix]

         
            scaled_logits = logits / temp

            probs = F.softmax(scaled_logits, dim=0)

            ix = torch.multinomial(
                probs,
                num_samples=1,
                replacement=True,
                generator=g
            ).item()

            if ix == 0:
                break

            out.append(itos[ix])

        print("".join(out))

In [6]:
sample_names(0.5)   # safer
sample_names(1.0)   # default
sample_names(2.0)   # wild


Temperature = 0.5
-------------------------
jamay
sadayrilein
daha
jon
arish

Temperature = 1.0
-------------------------
be
ka
nn
jo
s

Temperature = 2.0
-------------------------
t
arze
brghonhiymen
kpedeshemtazgawbe
deyli
